# Knee Exo — single-trial test

Minimal notebook for **one** knee-exo subject–trial pair.

- **Telemetry**: `os_kinetics/{trial_stem}.npz`
- **Mocap GPIO**: processed `{Subject}/knee-exo/mocap/{COND}_{speed}.csv` (`jet`)
- **OpenSim ID**: `{Subject}/knee-exo/id/{COND}_{speed}_id.sto` (`knee_angle_r_moment`)
- **Comparison (N·m/kg)**: blue = `ID/mass − model_out_nmpkg × scale`; black dotted = logged `model_out_nmpkg`
- **Diagnostics (N·m)**: OpenSim ID and `cmd_R` exo assist (causal LPF)

No model inference yet.

In [ ]:
import io
from pathlib import Path
from typing import Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
TELEMETRY_ROOT = PROJECT_ROOT

EXO_KIND = 'knee-exo'
MOCAP_FS_HZ = 1000.0

# Change this to inspect another trial.
TRIAL_STEM = 'ab04_changseob_knee_0p8mps_rd_exo_on'

SUBJECT_TOKEN_TO_DIR = {
    'ab01_jinwoo': 'AB01_Jinwoo', 'ab02_oscar': 'AB02_Oscar', 'ab03_ilseung': 'AB03_Ilseung',
    'ab04_changseob': 'AB04_Changseob', 'ab05_maria': 'AB05_Maria', 'ab06_jimin': 'AB06_Jimin',
    'ab07_amy': 'AB07_Amy', 'ab08_seokhyun': 'AB08_Seokhyun',
}
SUBJECT_MASS_KG = {
    'ab01_jinwoo': 88.0, 'ab02_oscar': 71.1, 'ab03_ilseung': 84.4, 'ab04_changseob': 74.0,
    'ab05_maria': 55.0, 'ab06_jimin': 82.6, 'ab07_amy': 51.3, 'ab08_seokhyun': 71.9,
}

EXO_KIND = 'knee-exo'
MOCAP_FS_HZ = 1000.0
MOMENT_COL = 'knee_angle_r_moment'
LPF_CUTOFF_HZ, LPF_ORDER = 6.0, 4
INPUT_LPF_MODE = 'causal'

GPIO_PALETTE = {'mocap': '#90A4AE', 'telemetry': '#FF9800'}

print(f'Trial: {TRIAL_STEM}')
print(f'Telemetry root: {TELEMETRY_ROOT}')
print(f'Processed root: {PROCESSED_ROOT}')

In [ ]:
def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    y = sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)
    return y


def lpf_nan(x, fs_hz, cutoff_hz, order, mode):
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite])
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def extract_model_out_nmpkg(npz) -> Tuple[np.ndarray, str]:
    for k in ('model_out_nmpkg', 'model_out_nmpkg_raw'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(
        f"No model output key in npz; expected model_out_nmpkg. Available: {sorted(npz.files)}"
    )


def infer_torque_scale(model_out_nmpkg: np.ndarray, moment_raw_nm: np.ndarray, mass_kg: float) -> float:
    """Runtime scale: moment_raw ≈ model_out_nmpkg × mass × scale."""
    denom = model_out_nmpkg * float(mass_kg)
    m = np.isfinite(denom) & np.isfinite(moment_raw_nm) & (np.abs(denom) > 1e-6)
    if not m.any():
        return 1.0
    return float(np.median(moment_raw_nm[m] / denom[m]))


def extract_applied_cmd_nm(npz) -> Tuple[np.ndarray, str]:
    """Exo assist command torque in N·m (cmd_R / cmd_L from telemetry)."""
    for k in ('cmd_R', 'cmd_L'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(
        f"No exo command key in npz; expected cmd_R or cmd_L. Available: {sorted(npz.files)}"
    )


def extract_moment_raw_nm(npz) -> Optional[np.ndarray]:
    """Logged pre-command motor torque for scale inference only."""
    if 'moment_raw' in npz.files:
        return np.asarray(npz['moment_raw'], dtype=np.float64)
    return None


def extract_applied_nm_knee(npz) -> Tuple[np.ndarray, str]:
    return extract_applied_cmd_nm(npz)


def extract_applied_r(npz) -> Tuple[np.ndarray, str]:
    return extract_applied_nm_knee(npz)


def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def _subject_token(stem: str) -> str:
    return '_'.join(stem.lower().split('_')[:2])


def subject_dir_from_stem(stem: str) -> Path:
    token = _subject_token(stem)
    name = SUBJECT_TOKEN_TO_DIR.get(token)
    if name is None:
        raise FileNotFoundError(f'Unknown subject token: {token}')
    p = PROCESSED_ROOT / name
    if not p.is_dir():
        raise FileNotFoundError(f'Processed subject folder missing: {p}')
    return p


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS_HZ):
    """Parse Vicon analog CSV (5-row header); return (time [s], jet GPIO [V])."""
    df = pd.read_csv(
        path, skiprows=[0, 1, 2, 4], header=0,
        low_memory=False, on_bad_lines='skip',
    )
    df = df[pd.to_numeric(df['Frame'], errors='coerce').notna()].copy()
    df['jet'] = pd.to_numeric(df['jet'], errors='coerce')
    df = df.dropna(subset=['jet']).reset_index(drop=True)
    time = np.arange(len(df)) / fs
    return time, df['jet'].to_numpy(dtype=float)


def normalize_gpio(gpio: np.ndarray) -> np.ndarray:
    arr = np.asarray(gpio, dtype=np.float64)
    g_range = arr.max() - arr.min()
    if g_range <= 0:
        return arr
    return (arr - arr.min()) / g_range


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> Optional[int]:
    """Index of the first sample immediately after a falling edge."""
    above = np.asarray(signal, dtype=np.float64) > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


def extract_gpio(npz) -> Tuple[np.ndarray, str]:
    for k in ('gpio_output', 'GPIO', 'gpio', 'trigger'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError('No GPIO key in npz')


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_falling_edge(g_exo)
    idx_mocap = first_falling_edge(normalize_gpio(g_mocap))
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def resolve_trial_paths(trial_stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(trial_stem)
    subj_dir = subject_dir_from_stem(trial_stem)
    npz_path = TELEMETRY_ROOT / f'{trial_stem}.npz'
    mocap_path = subj_dir / EXO_KIND / 'mocap' / f'{cond}_{speed}.csv'
    return {
        'npz': npz_path,
        'mocap': mocap_path,
        'cond': cond,
        'speed': speed,
        'subject_dir': subj_dir,
    }


def load_gpio_sync_data(trial_stem: str) -> Dict:
    paths = resolve_trial_paths(trial_stem)
    for key in ('npz', 'mocap'):
        if not paths[key].exists():
            raise FileNotFoundError(f'Missing {key} file: {paths[key]}')

    d = np.load(str(paths['npz']), allow_pickle=True)
    gpio, gpio_key = extract_gpio(d)
    t_raw = (
        np.asarray(d['time'], dtype=np.float64)
        if 'time' in d.files else np.arange(len(gpio), dtype=np.float64)
    )
    n = min(len(t_raw), len(gpio))
    t_raw, gpio = t_raw[:n], gpio[:n]

    t_mocap, gpio_mocap = parse_mocap_csv(paths['mocap'])
    gpio_mocap_norm = normalize_gpio(gpio_mocap)
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)

    return {
        'trial': trial_stem,
        'paths': paths,
        'gpio_key': gpio_key,
        'offset_s': offset_s,
        'idx_exo': idx_exo,
        'idx_mocap': idx_mocap,
        'fs_npz_hz': infer_fs_hz(t_raw),
        'fs_mocap_hz': infer_fs_hz(t_mocap, default_fs=MOCAP_FS_HZ),
        't_npz': t_raw,
        'gpio_npz': gpio,
        't_mocap': t_mocap,
        'gpio_mocap_raw': gpio_mocap,
        'gpio_mocap_norm': gpio_mocap_norm,
        't_npz_aligned': t_raw + float(offset_s) if offset_s is not None else t_raw.copy(),
    }


def load_moment_waveforms(trial_stem: str, sync: Dict) -> Dict:
    """Load GPIO-synced waveforms.

    OpenSim ID and exo cmd (cmd_R/cmd_L) get a causal 6 Hz LPF at load time.
    Logged `model_out_nmpkg` is used as-is (already output-LPF'd on device).
    """
    paths = sync['paths']
    cond, speed = paths['cond'], paths['speed']
    id_path = paths['subject_dir'] / EXO_KIND / 'id' / f'{cond}_{speed}_id.sto'
    if not id_path.exists():
        raise FileNotFoundError(f'Missing ID file: {id_path}')

    mass = SUBJECT_MASS_KG[_subject_token(trial_stem)]
    d = np.load(str(paths['npz']), allow_pickle=True)
    applied_nm, applied_key = extract_applied_cmd_nm(d)
    moment_raw_nm = extract_moment_raw_nm(d)
    model_out_raw, model_out_key = extract_model_out_nmpkg(d)

    n = min(len(sync['t_npz']), len(applied_nm), len(model_out_raw))
    t_aligned = sync['t_npz_aligned'][:n].astype(np.float64)
    applied_nm = applied_nm[:n]
    model_out_raw = model_out_raw[:n]
    fs_hz = infer_fs_hz(sync['t_npz'][:n])

    scale_src = moment_raw_nm[:n] if moment_raw_nm is not None else applied_nm
    torque_scale = infer_torque_scale(model_out_raw, scale_src, mass)

    cols, id_data = read_sto(id_path)
    t_id = id_data[:, cols.index('time')]
    id_moment_nm = id_data[:, cols.index(MOMENT_COL)]
    fs_id = infer_fs_hz(t_id)

    id_moment_lpf = butter_lpf(id_moment_nm, fs_id, LPF_CUTOFF_HZ, LPF_ORDER, INPUT_LPF_MODE)
    id_nm = np.interp(t_aligned, t_id, id_moment_lpf, left=np.nan, right=np.nan)
    applied_nm_lpf = butter_lpf(applied_nm, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, INPUT_LPF_MODE)

    model_out_nmpkg = np.asarray(model_out_raw, dtype=np.float64)
    id_minus_model_nmpkg = id_nm / mass - model_out_nmpkg * torque_scale

    return {
        'trial': trial_stem,
        't': t_aligned,
        'id_nm': id_nm,
        'applied_nm': applied_nm_lpf,
        'id_minus_model_nmpkg': id_minus_model_nmpkg,
        'model_out_nmpkg': model_out_nmpkg,
        'torque_scale': torque_scale,
        'applied_key': applied_key,
        'model_out_key': model_out_key,
        'mass_kg': mass,
        'id_path': id_path,
        'moment_col': MOMENT_COL,
        'fs_hz': fs_hz,
    }


print('GPIO helpers ready.')

In [ ]:
SYNC = load_gpio_sync_data(TRIAL_STEM)
paths = SYNC['paths']

if SYNC['offset_s'] is None:
    raise RuntimeError(
        f'No GPIO falling edge (telemetry idx={SYNC["idx_exo"]}, mocap idx={SYNC["idx_mocap"]})'
    )

print(f"NPZ:   {paths['npz']}")
print(f"Mocap: {paths['mocap']}  ({paths['cond']}_{paths['speed']})")
print(f"GPIO key: {SYNC['gpio_key']}")
print(f"Sample rates: telemetry={SYNC['fs_npz_hz']:.1f} Hz, mocap={SYNC['fs_mocap_hz']:.1f} Hz")
print(f"Telemetry GPIO range: [{SYNC['gpio_npz'].min():.3f}, {SYNC['gpio_npz'].max():.3f}]")
print(f"Mocap jet range:      [{SYNC['gpio_mocap_raw'].min():.3f}, {SYNC['gpio_mocap_raw'].max():.3f}] V")
print()
print(f"Falling edge idx: telemetry={SYNC['idx_exo']}, mocap={SYNC['idx_mocap']}")
print(f"Falling edge time: telemetry={SYNC['t_npz'][SYNC['idx_exo']]:.4f}s, mocap={SYNC['t_mocap'][SYNC['idx_mocap']]:.4f}s")
print(f"Applied offset (mocap - telemetry): {SYNC['offset_s']:+.6f} s")
print(f"Aligned edge check: {SYNC['t_npz_aligned'][SYNC['idx_exo']]:.6f}s vs {SYNC['t_mocap'][SYNC['idx_mocap']]:.6f}s")

In [ ]:
def draw_gpio_sync(sync: Dict, window_s: float = 4.0) -> None:
    if sync['offset_s'] is None:
        raise RuntimeError('Sync data has no falling edge')

    t_edge_mocap = float(sync['t_mocap'][sync['idx_mocap']])
    t_edge_npz = float(sync['t_npz'][sync['idx_exo']])
    t_edge_aligned = float(sync['t_npz_aligned'][sync['idx_exo']])
    t0, t1 = t_edge_mocap - 0.5, t_edge_mocap + window_s

    fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

    m_before_npz = (sync['t_npz'] >= t_edge_npz - 0.5) & (sync['t_npz'] <= t_edge_npz + window_s)
    m_before_mocap = (sync['t_mocap'] >= t0) & (sync['t_mocap'] <= t1)
    axs[0].plot(
        sync['t_npz'][m_before_npz], sync['gpio_npz'][m_before_npz],
        color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--', label=f"Telemetry ({sync['gpio_key']})",
    )
    axs[0].plot(
        sync['t_mocap'][m_before_mocap], sync['gpio_mocap_norm'][m_before_mocap],
        color=GPIO_PALETTE['mocap'], lw=1.2, alpha=0.85, label='Mocap jet (normalised)',
    )
    axs[0].axvline(t_edge_npz, color=GPIO_PALETTE['telemetry'], ls=':', lw=1.2, alpha=0.9)
    axs[0].axvline(t_edge_mocap, color=GPIO_PALETTE['mocap'], ls=':', lw=1.2, alpha=0.9)
    axs[0].set_ylabel('Amplitude (a.u.)')
    axs[0].set_title(
        f'Before sync — falling edges at {t_edge_npz:.3f}s (telemetry) vs {t_edge_mocap:.3f}s (mocap)'
    )
    axs[0].legend(loc='upper right')
    axs[0].grid(alpha=0.25)

    m_after_npz = (sync['t_npz_aligned'] >= t0) & (sync['t_npz_aligned'] <= t1)
    m_after_mocap = (sync['t_mocap'] >= t0) & (sync['t_mocap'] <= t1)
    axs[1].plot(
        sync['t_mocap'][m_after_mocap], sync['gpio_mocap_norm'][m_after_mocap],
        color=GPIO_PALETTE['mocap'], lw=1.2, alpha=0.85, label='Mocap jet (normalised)',
    )
    axs[1].plot(
        sync['t_npz_aligned'][m_after_npz], sync['gpio_npz'][m_after_npz],
        color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--',
        label=f"Telemetry shifted ({sync['gpio_key']})",
    )
    axs[1].axvline(t_edge_mocap, color='black', ls=':', lw=1.4, alpha=0.8, label='Falling edge')
    axs[1].set_xlabel('Mocap time (s)')
    axs[1].set_ylabel('Amplitude (a.u.)')
    axs[1].set_title(
        f'After sync — offset {sync["offset_s"]:+.4f}s | aligned edge {t_edge_aligned:.4f}s'
    )
    axs[1].legend(loc='upper right')
    axs[1].grid(alpha=0.25)

    fig.suptitle(f"{sync['trial']} | GPIO sync pulse", y=1.01, fontsize=12)
    fig.tight_layout()
    plt.show()


draw_gpio_sync(SYNC, window_s=4.0)

## Moment waveforms (GPIO-synced, 6 Hz zero-phase LPF)

**Left axis (N·m):** OpenSim ID (gray), exo assist `cmd_R` (red, causal LPF)

**Right axis (N·m/kg):**
- **Blue solid**: `ID/mass − model_out_nmpkg × scale`
- **Black dotted**: logged `model_out_nmpkg`

In [ ]:
WAVE = load_moment_waveforms(TRIAL_STEM, SYNC)

print(f"ID file: {WAVE['id_path']}")
print(f"Moment column: {WAVE['moment_col']}")
print(f"Applied torque key: {WAVE['applied_key']}")
print(f"Subject mass: {WAVE['mass_kg']:.1f} kg")
print(f"Telemetry sample rate: {WAVE['fs_hz']:.1f} Hz")
print(f"Duration: {WAVE['t'][-1] - WAVE['t'][0]:.1f} s")
print(f"Model output key: {WAVE['model_out_key']}")
print(f"Inferred torque scale: {WAVE['torque_scale']:.4f}")
print(
    f"ID range:           [{np.nanmin(WAVE['id_nm']):+.2f}, {np.nanmax(WAVE['id_nm']):+.2f}] N·m"
)
print(
    f"Applied range:      [{np.nanmin(WAVE['applied_nm']):+.2f}, {np.nanmax(WAVE['applied_nm']):+.2f}] N·m"
)
print(
    f"ID−model×scale:     [{np.nanmin(WAVE['id_minus_model_nmpkg']):+.3f}, {np.nanmax(WAVE['id_minus_model_nmpkg']):+.3f}] N·m/kg"
)
print(
    f"model_out_nmpkg:    [{np.nanmin(WAVE['model_out_nmpkg']):+.3f}, {np.nanmax(WAVE['model_out_nmpkg']):+.3f}] N·m/kg"
)

In [ ]:
wave_out = widgets.Output()
time_slider = widgets.FloatRangeSlider(
    description='Time (s):',
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='760px'),
    style={'description_width': 'initial'},
)


def draw_moment_waveforms(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    t0, t1 = t_window
    m = (
        (t_rel >= t0) & (t_rel <= t1)
        & np.isfinite(wave['id_nm'])
        & np.isfinite(wave['applied_nm'])
        & np.isfinite(wave['id_minus_model_nmpkg'])
        & np.isfinite(wave['model_out_nmpkg'])
    )

    fig, ax_nm = plt.subplots(figsize=(14, 5))
    ax_kg = ax_nm.twinx()

    ax_nm.plot(
        t_rel[m], wave['id_nm'][m],
        color='#9e9e9e', lw=1.6, ls='-',
        label=f"OpenSim ID ({wave['moment_col']})",
    )
    ax_nm.plot(
        t_rel[m], wave['applied_nm'][m],
        color='#e53935', lw=2.0, ls='-',
        label=f"Exo assist ({wave['applied_key']})",
    )
    ax_kg.plot(
        t_rel[m], wave['id_minus_model_nmpkg'][m],
        color='#1e88e5', lw=2.0, ls='-',
        label=f"ID/mass − {wave['model_out_key']} × scale",
    )
    ax_kg.plot(
        t_rel[m], wave['model_out_nmpkg'][m],
        color='black', lw=1.8, ls=':',
        label=f"Logged {wave['model_out_key']}",
    )

    ax_nm.set_xlabel('Time since trial start (s)')
    ax_nm.set_ylabel('Torque (N·m)')
    ax_kg.set_ylabel('Moment (N·m/kg)')
    ax_nm.set_title(
        f"{wave['trial']} | mass={wave['mass_kg']:.1f} kg | scale={wave['torque_scale']:.3f} | "
        f"{LPF_CUTOFF_HZ:.0f} Hz {LPF_ORDER}-ord causal LPF (ID/cmd); logged model_out"
    )
    ax_nm.grid(alpha=0.25)

    lines_nm, labels_nm = ax_nm.get_legend_handles_labels()
    lines_kg, labels_kg = ax_kg.get_legend_handles_labels()
    ax_nm.legend(lines_nm + lines_kg, labels_nm + labels_kg, loc='upper right')

    fig.tight_layout()
    with wave_out:
        wave_out.clear_output(wait=True)
        plt.show()


def _init_time_slider(wave: Dict) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    time_slider.min = float(t_rel[0])
    time_slider.max = float(t_rel[-1])
    time_slider.step = max((time_slider.max - time_slider.min) / 500, 1e-3)
    time_slider.value = (time_slider.min, time_slider.max)


def _redraw_waveforms(*_) -> None:
    draw_moment_waveforms(WAVE, time_slider.value)


time_slider.observe(_redraw_waveforms, names='value')
_init_time_slider(WAVE)
display(widgets.VBox([time_slider, wave_out]))
_redraw_waveforms()

## Model vs ID-derived moment (N·m/kg)

Same comparison on a single axis:

- **Blue solid**: `ID/mass − model_out_nmpkg × scale`
- **Black dotted**: logged `model_out_nmpkg`

In [ ]:
def draw_net_moment_per_mass(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    t0, t1 = t_window
    m = (
        (t_rel >= t0) & (t_rel <= t1)
        & np.isfinite(wave['id_minus_model_nmpkg'])
        & np.isfinite(wave['model_out_nmpkg'])
    )

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        t_rel[m], wave['id_minus_model_nmpkg'][m],
        color='#1e88e5', lw=2.0, ls='-',
        label=f"ID/mass − {wave['model_out_key']} × scale",
    )
    ax.plot(
        t_rel[m], wave['model_out_nmpkg'][m],
        color='black', lw=1.8, ls=':',
        label=f"Logged {wave['model_out_key']}",
    )

    ax.set_xlabel('Time since trial start (s)')
    ax.set_ylabel('Moment (N·m/kg)')
    ax.set_title(
        f"{wave['trial']} | mass={wave['mass_kg']:.1f} kg | scale={wave['torque_scale']:.3f} | "
        f"{LPF_CUTOFF_HZ:.0f} Hz {LPF_ORDER}-ord causal LPF (ID/cmd); logged model_out"
    )
    ax.axhline(0.0, color='gray', lw=0.6, ls=':')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper right')
    fig.tight_layout()

    with net_out:
        net_out.clear_output(wait=True)
        plt.show()


net_out = widgets.Output()
net_time_slider = widgets.FloatRangeSlider(
    description='Time (s):',
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='760px'),
    style={'description_width': 'initial'},
)


def _init_net_time_slider(wave: Dict) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    net_time_slider.min = float(t_rel[0])
    net_time_slider.max = float(t_rel[-1])
    net_time_slider.step = max((net_time_slider.max - net_time_slider.min) / 500, 1e-3)
    net_time_slider.value = (net_time_slider.min, net_time_slider.max)


def _redraw_net_moment(*_) -> None:
    draw_net_moment_per_mass(WAVE, net_time_slider.value)


net_time_slider.observe(_redraw_net_moment, names='value')
_init_net_time_slider(WAVE)
display(widgets.VBox([net_time_slider, net_out]))
_redraw_net_moment()